In [175]:
#!pip install torchmetrics

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
#from torchvision import datasets
from sklearn import datasets
#from torchvision.transforms import ToTensor, Lambda, Compose
from torchmetrics import F1Score, Accuracy
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

In [167]:
data= datasets.load_wine()
Xtrain,Xtest,ytrain, ytest=train_test_split(data.data, data.target, test_size=0.3, random_state=1, shuffle=True)
print(len(Xtrain), len(Xtest))
train, test=({'X':Xtrain,'y':ytrain}, {'X':Xtest,'y':ytest})
class model(nn.Module):
  def __init__(self,in_features, out_features):
    super().__init__()
    self.in_features=in_features
    self.out_features=out_features
    self.Linear=nn.Linear(self.in_features, self.out_features)

  def forward(self, x):
    y_pred= self.Linear(x)
    return y_pred

class Data(Dataset):
  def __init__(self, sample):
    self.sample = sample
    self.x = sample['X'].astype(np.float32)
    self.y = sample['y'].astype(np.float32)

  def __getitem__(self, index):
    return (self.x[index], self.y[index])

  def __len__(self):
    return len(self.x)

124 54


In [ ]:
train_data=Data(train)
test_data=Data(test)
train_data=DataLoader(train_data, batch_size=10,shuffle=True)
test_data= DataLoader(test_data, batch_size=10,shuffle=True)
for batch in train_data:
    print(batch)
print(len(train_data))

In [ ]:
epochs=1000
model1=model(13,3)
criterion=nn.CrossEntropyLoss()
optim=torch.optim.Adam(model1.parameters(), lr=0.001)
for epoch in range(epochs):
  for i, (data1, label) in enumerate(train_data):
    label = label.long()
    y_pred=model1(data1)
    loss=criterion(y_pred, label)
    loss.backward()
    optim.step()
    optim.zero_grad()
    if epoch%10==0:
      print(loss.detach())

In [190]:
pred =[]
with torch.inference_mode():
    for i, (data1, label) in enumerate(test_data):
        label = label.long()
        ypred= model1(data1)
        soft=nn.Softmax(dim=1)
        pred=soft(ypred).argmax(axis=-1)
        print(f'pred: {pred},label: {label}')
        print((pred==label).sum())
accu=Accuracy(task="multiclass", num_classes=3, average='macro')


pred: tensor([2, 0, 0, 2, 1, 2, 1, 1, 1, 2]),label: tensor([2, 0, 0, 2, 1, 2, 1, 2, 1, 2])
tensor(9)
pred: tensor([1, 1, 1, 2, 0, 0, 0, 0, 1, 0]),label: tensor([1, 1, 1, 2, 0, 0, 0, 0, 1, 0])
tensor(10)
pred: tensor([1, 0, 0, 1, 0, 0, 0, 1, 0, 1]),label: tensor([1, 0, 0, 1, 0, 0, 0, 1, 0, 0])
tensor(9)
pred: tensor([1, 0, 2, 2, 1, 0, 1, 1, 1, 1]),label: tensor([1, 0, 2, 2, 1, 0, 1, 1, 1, 1])
tensor(10)
pred: tensor([2, 1, 1, 0, 1, 0, 2, 0, 0, 0]),label: tensor([2, 2, 1, 0, 1, 0, 2, 0, 0, 0])
tensor(9)
pred: tensor([0, 0, 1, 2]),label: tensor([0, 0, 1, 2])
tensor(4)


(178, 13)